In [1]:
import requests
import numpy as np
import spiceypy as sp

from src.utils.units_and_conversions import arcsec_to_rad
from main import main

In [2]:
x, t_start, t_end, residual_states = main()

In [3]:
print(np.sqrt(np.mean(residual_states**2)) / arcsec_to_rad)
print(np.percentile(np.abs(residual_states) / arcsec_to_rad, [50, 90, 95, 99]))

mean_residual = np.mean(np.abs(residual_states)) / arcsec_to_rad
print(f"Mean Residual in arcseconds: {mean_residual}")

0.1685491338661102
[0.0794417  0.21405147 0.28847597 0.60736559]
Mean Residual in arcseconds: 0.110438258227337


In [4]:
command = "&COMMAND='433'"
obj_data = "&OBJ_DATA='YES'"
make_ephem = "&MAKE_EPHEM='YES'"
ephem_type = "&EPHEM_TYPE='VECTORS'"
center = "&CENTER='@10'" # Sun body center / heliocentric
ref_plane = "&REF_PLANE='F'" # ICRF/J2000 equatorial
out_units = "&OUT_UNITS='KM-S'"
start_time = f"&START_TIME='JD {sp.unitim(t_start, 'ET', 'JDTDB'):.12f}'"
stop_time = f"&STOP_TIME='JD {sp.unitim(t_start + 86400.0, 'ET', 'JDTDB'):.12f}'"
step_size = "&STEP_SIZE='1d'"

response = requests.get(f"https://ssd.jpl.nasa.gov/api/horizons.api?format=json&CSV_FORMAT='YES'{command}{obj_data}{make_ephem}{ephem_type}{center}{ref_plane}{out_units}{start_time}{stop_time}{step_size}")

In [5]:
result_text = response.json()['result']
print(result_text)

*******************************************************************************
JPL/HORIZONS                 433 Eros (A898 PA)            2026-Jul-20 12:19:46
Rec #:     433 (+COV) Soln.date: 2021-May-24_17:55:05   # obs: 9130 (1893-2021)
 
IAU76/J2000 helio. ecliptic osc. elements (au, days, deg., period=Julian yrs):
 
  EPOCH=  2453311.5 ! 2004-Nov-02.00 (TDB)         Residual RMS= .3494
   EC= .2228078944584026   QR= 1.133355399799004   TP= 2453371.5859943051
   OM= 304.4010273379536   W=  178.665326776373    IN= 10.8291838260782
   A= 1.458269315549994    MA= 326.37047603642     ADIST= 1.783183231300984
   PER= 1.76102            N= .559689897           ANGMOM= .020250867
   DAN= 1.78304            DDN= 1.13341            L= 123.090114
   B= .2507387             MOID= .149124           TP= 2005-Jan-01.0859943051
 
Asteroid physical parameters (km, seconds, rotational period in hours):
   GM= .0004463            RAD= 8.42               ROTPER= 5.27
   H= 10.4                 G= .46

In [6]:
import pandas as pd
from io import StringIO

csv_block = result_text.split("$$SOE")[1].split("$$EOE")[0].strip()

cols = ["JDTDB", "date", "x", "y", "z", "vx", "vy", "vz", "lt", "rg", "rr"]
df = pd.read_csv(StringIO(csv_block), header=None, usecols=range(11), names=cols)

In [7]:
x_jpl = []
for col in cols[2:8]:
    v = df[col][0]
    x_jpl.append(v.item())

In [8]:
print([b-a for a,b in zip(x, x_jpl)])

[np.float64(-5.989882469177246), np.float64(-4.906225327402353), np.float64(20.18419274315238), np.float64(1.2836560989271106e-05), np.float64(1.3093722742496539e-05), np.float64(-3.054618286313371e-06)]
